# CNN model (updated)

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from quickdraw import QuickDrawDataGroup

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

In [2]:
SEED = 42
SAMPLES_PER_CATEGORY = 3000
IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
EMBEDDING_SIZE = 128

CATEGORY_FILE = Path("../../data/categories/objects.txt")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Load category names

In [3]:
with open(CATEGORY_FILE, "r") as f:
    text = f.read()

categories = [category.strip() for category in text.replace("\n", ",").split(",") if category.strip()]

label_map = {category: index for index, category in enumerate(categories)}
reverse_label_map = {index: category for category, index in label_map.items()}

print("Number of categories:", len(categories))
print(categories)

Number of categories: 15
['backpack', 'bed', 'book', 'chair', 'clock', 'cup', 'door', 'key', 'knife', 'laptop', 'shoe', 'spoon', 'table', 'television', 'toothbrush']


## Convert drawings into image arrays

In [4]:
def drawing_to_array(drawing, image_size=IMAGE_SIZE):
    image = drawing.image
    image = image.convert("L")
    image = image.resize((image_size, image_size), Image.Resampling.LANCZOS)
    
    array = np.array(image).astype(np.float32)
    array = 255.0 - array
    array = array / 255.0 # Normalize from 0-255 to 0-1
    
    return array

## Build dataset


In [5]:
X = []
y = []

for category in categories:
    print(f"Loading {category}...")
    group = QuickDrawDataGroup(
        category,
        max_drawings=SAMPLES_PER_CATEGORY,
        print_messages=False
    )
    
    count = 0
    for drawing in group.drawings:
        array = drawing_to_array(drawing, IMAGE_SIZE)
        X.append(array)
        y.append(label_map[category])
        count += 1
        if count >= SAMPLES_PER_CATEGORY:
            break

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)

Loading backpack...
Loading bed...
Loading book...
Loading chair...
Loading clock...
Loading cup...
Loading door...
Loading key...
Loading knife...
Loading laptop...
Loading shoe...
Loading spoon...
Loading table...
Loading television...
Loading toothbrush...
X shape: (45000, 128, 128)
y shape: (45000,)


## Prepare CNN data

In [6]:
# Add channel dimension: (samples, 1, 128, 128)
X_cnn = X[:, np.newaxis, :, :]

X_train, X_test, y_train, y_test = train_test_split(
    X_cnn,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training data:", X_train_tensor.shape)
print("Testing data:", X_test_tensor.shape)

Training data: torch.Size([36000, 1, 128, 128])
Testing data: torch.Size([9000, 1, 128, 128])


## Define the CNN model

This simple CNN has:

- 2 convolution layers
- 2 max pooling layers
- 1 flatten layer
- 1 embedding layer that creates a 128-dimensional feature vector
- 1 final classification layer

In [7]:
class EmbeddingObjectCNN(nn.Module):
    def __init__(self, num_classes, embedding_size=128):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten()
        )

        # 128x128 becomes 64x64 after first pooling,
        # then 32x32 after second pooling.
        self.embedding = nn.Sequential(
            nn.Linear(32 * 32 * 32, embedding_size),
            nn.ReLU()
        )
        self.classifier = nn.Linear(embedding_size, num_classes)

    def forward(self, x):
        features = self.features(x)
        embedding = self.embedding(features)
        output = self.classifier(embedding)
        return output

    def get_embedding(self, x):
        features = self.features(x)
        embedding = self.embedding(features)
        return embedding

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EmbeddingObjectCNN(
    num_classes=len(categories),
    embedding_size=EMBEDDING_SIZE
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [26]:
'''
MODEL_DIR = Path("../../data/models")

model = EmbeddingObjectCNN(
    num_classes=len(categories),
    embedding_size=EMBEDDING_SIZE
).to(device)

model.load_state_dict(
    torch.load(
        MODEL_DIR / "objects_cnn_embedding_model.pt",
        map_location=device
    )
)

model.eval()

print("Loaded trained model.")
'''

Loaded trained model.


## Train the CNN

In [9]:
train_losses = []
train_accuracies = []
test_accuracies = []

best_accuracy = 0
best_epoch = 0
best_model_state = None

for epoch in range(EPOCHS):

    # Training
    model.train()
    
    total_loss = 0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        train_predictions = torch.argmax(outputs, dim=1)
        train_correct += (train_predictions == labels).sum().item()
        train_total += labels.size(0)
    
    average_loss = total_loss / len(train_loader)
    train_accuracy = train_correct / train_total
    
    # Testing
    model.eval()
    
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            test_predictions = torch.argmax(outputs, dim=1)
            
            test_correct += (test_predictions == labels).sum().item()
            test_total += labels.size(0)
    
    test_accuracy = test_correct / test_total
    
    train_losses.append(average_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)
    
    # Save the best model based on test accuracy
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        best_epoch = epoch + 1
        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }
    
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {average_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Test Accuracy: {test_accuracy:.4f}"
    )

# Load the best model after training finishes
model.load_state_dict(best_model_state)

cnn_accuracy = best_accuracy

print("\nTraining complete!")
print(f"Best epoch: {best_epoch}")
print(f"Best CNN test accuracy: {cnn_accuracy:.4f}")
print(f"Best CNN test accuracy percentage: {cnn_accuracy * 100:.2f}%")


Epoch 1/10 | Loss: 0.9379 | Train Accuracy: 0.7183 | Test Accuracy: 0.8109
Epoch 2/10 | Loss: 0.4582 | Train Accuracy: 0.8595 | Test Accuracy: 0.8167
Epoch 3/10 | Loss: 0.2680 | Train Accuracy: 0.9166 | Test Accuracy: 0.8122
Epoch 4/10 | Loss: 0.1389 | Train Accuracy: 0.9551 | Test Accuracy: 0.8181
Epoch 5/10 | Loss: 0.0838 | Train Accuracy: 0.9727 | Test Accuracy: 0.8072
Epoch 6/10 | Loss: 0.0602 | Train Accuracy: 0.9796 | Test Accuracy: 0.8063
Epoch 7/10 | Loss: 0.0450 | Train Accuracy: 0.9853 | Test Accuracy: 0.8063
Epoch 8/10 | Loss: 0.0350 | Train Accuracy: 0.9890 | Test Accuracy: 0.8059
Epoch 9/10 | Loss: 0.0357 | Train Accuracy: 0.9880 | Test Accuracy: 0.8083
Epoch 10/10 | Loss: 0.0282 | Train Accuracy: 0.9908 | Test Accuracy: 0.8081

Training complete!
Best epoch: 4
Best CNN test accuracy: 0.8181
Best CNN test accuracy percentage: 81.81%


In [34]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("Loaded model accuracy:", correct / total)

Loaded model accuracy: 0.8181111111111111


Saved best trained model to: ../../data/models/objects_cnn_embedding_model.pt


## Extract embeddings from CNN

Runs all dataset images through the CNN and extracts the 128-dimensional embeddings from the layer before classification.

In [36]:
# Prepare the full dataset for embedding extraction
X_all_cnn = X[:, np.newaxis, :, :]
X_all_tensor = torch.tensor(X_all_cnn, dtype=torch.float32)

all_dataset = TensorDataset(X_all_tensor, torch.tensor(y, dtype=torch.long))
all_loader = DataLoader(all_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_embeddings = []
all_embedding_labels = []

model.eval()

with torch.no_grad():
    for images, labels in all_loader:
        images = images.to(device)

        embeddings = model.get_embedding(images)

        all_embeddings.append(embeddings.cpu().numpy())
        all_embedding_labels.append(labels.numpy())

all_embeddings = np.vstack(all_embeddings)
all_embedding_labels = np.concatenate(all_embedding_labels)

print("Embeddings shape:", all_embeddings.shape)
print("Labels shape:", all_embedding_labels.shape)

Embeddings shape: (45000, 128)
Labels shape: (45000,)


## Compute class centroids k=1,5,10

In [37]:
from sklearn.cluster import KMeans

centroids_by_k = {}

for k in [1, 5, 10]:
    centroids_by_k[k] = {}

    for class_index, class_name in reverse_label_map.items():
        class_embeddings = all_embeddings[all_embedding_labels == class_index]

        if k == 1:
            centroids = class_embeddings.mean(axis=0, keepdims=True)
        else:
            kmeans = KMeans(
                n_clusters=k,
                random_state=SEED,
                n_init=10
            )
            kmeans.fit(class_embeddings)
            centroids = kmeans.cluster_centers_

        centroids_by_k[k][class_name] = centroids

    print(f"Finished k={k} centroids")

Finished k=1 centroids
Finished k=5 centroids
Finished k=10 centroids


In [38]:
CENTROIDS_DIR = Path("../../data/centroids")
CENTROIDS_DIR.mkdir(parents=True, exist_ok=True)

for k, centroid_dict in centroids_by_k.items():
    save_path = CENTROIDS_DIR / f"objects_{k}.npz"
    
    np.savez(
        save_path,
        centroids=centroid_dict,
        categories=np.array(categories),
        embedding_size=EMBEDDING_SIZE,
        image_size=IMAGE_SIZE,
        samples_per_category=SAMPLES_PER_CATEGORY,
        best_accuracy=cnn_accuracy
    )
    
    print("Saved:", save_path)

Saved: ../../data/centroids/objects_1.npz
Saved: ../../data/centroids/objects_5.npz
Saved: ../../data/centroids/objects_10.npz


## Load good and bad evaluation drawings

In [39]:
BASE_DRAWINGS_DIR = Path("../../data/base_drawings/objects")

In [40]:
drawing_paths = {}

for class_name in categories:
    for quality in ["good", "bad"]:
        path = BASE_DRAWINGS_DIR / f"{class_name}_{quality}.jpg"
        drawing_paths[(class_name, quality)] = path

In [41]:
def load_eval_image(image_path, image_size=IMAGE_SIZE):
    image = Image.open(image_path).convert("L")
    image = image.resize((image_size, image_size), Image.Resampling.LANCZOS)

    array = np.array(image).astype(np.float32)
    array = 255.0 - array
    array = array / 255.0 # Normalize to 0-1
    array = array[np.newaxis, np.newaxis, :, :] # Add CNN dimensions: (1, 1, 128, 128)

    return torch.tensor(array, dtype=torch.float32)

In [42]:
eval_embeddings = []

model.eval()

with torch.no_grad():
    for class_name in categories:
        for quality in ["good", "bad"]:
            image_path = drawing_paths[(class_name, quality)]
            
            image_tensor = load_eval_image(image_path, IMAGE_SIZE).to(device)
            
            embedding = model.get_embedding(image_tensor)
            embedding = embedding.cpu().numpy()
            
            eval_embeddings.append({
                "class": class_name,
                "quality": quality,
                "file_path": str(image_path),
                "embedding": embedding
            })

print("Number of eval embeddings:", len(eval_embeddings))

Number of eval embeddings: 30


## cosine similarity report

In [43]:
report_rows = []

for item in eval_embeddings:
    class_name = item["class"]
    quality = item["quality"]
    drawing_embedding = item["embedding"]

    for k in [1, 5, 10]:
        centroids = centroids_by_k[k][class_name]

        similarities = cosine_similarity(drawing_embedding, centroids)

        average_similarity = similarities.mean()
        max_similarity = similarities.max()

        report_rows.append({
            "class": class_name,
            "quality": quality,
            "k_centroids": k,
            "average_cosine_similarity": round(float(average_similarity), 4),
            "max_cosine_similarity": round(float(max_similarity), 4)
        })

cosine_report = pd.DataFrame(report_rows)

print("Report shape:", cosine_report.shape)

results = {}

for quality in ["good", "bad"]:
    quality_df = cosine_report[cosine_report["quality"] == quality]

    results[quality] = quality_df.pivot(
        index="k_centroids",
        columns="class",
        values="average_cosine_similarity"
    )

print("good drawings")
display(results["good"].round(3))

print("\nbad drawings")
display(results["bad"].round(3))

Report shape: (90, 5)
good drawings


class,backpack,bed,book,chair,clock,cup,door,key,knife,laptop,shoe,spoon,table,television,toothbrush
k_centroids,,,,,,,,,,,,,,,
1,0.308,0.398,0.258,0.498,0.282,0.356,0.316,0.357,0.253,0.584,0.439,0.330,0.300,0.506,0.195
5,0.296,0.358,0.230,0.472,0.277,0.336,0.320,0.334,0.222,0.550,0.390,0.309,0.306,0.482,0.154
10,0.296,0.349,0.222,0.453,0.275,0.340,0.322,0.309,0.220,0.533,0.416,0.289,0.311,0.478,0.156



bad drawings


class,backpack,bed,book,chair,clock,cup,door,key,knife,laptop,shoe,spoon,table,television,toothbrush
k_centroids,,,,,,,,,,,,,,,
1,0.550,0.441,0.345,0.729,0.304,0.250,0.436,0.286,0.466,0.671,0.449,0.524,0.394,0.624,0.260
5,0.526,0.423,0.304,0.677,0.298,0.239,0.424,0.275,0.449,0.630,0.398,0.470,0.386,0.567,0.212
10,0.523,0.393,0.294,0.644,0.299,0.246,0.421,0.250,0.424,0.613,0.415,0.459,0.384,0.565,0.212


In [49]:
'''
import gradio as gr

def predict(editor_value):
    if editor_value is None:
        return pd.DataFrame()

    img = editor_value["composite"]

    if img is None:
        return pd.DataFrame()

    # Same preprocessing used for QuickDraw training images
    # Preprocess Gradio drawing
    img = img.convert("RGB")
    img = img.resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        Image.Resampling.LANCZOS
    )
    
    img_array = np.array(img).astype(np.float32)
    
    # Convert colored/black strokes on white background
    # into white strokes on black background
    img_array = 1.0 - (img_array.min(axis=2) / 255.0)
    
    input_tensor = torch.tensor(
        img_array,
        dtype=torch.float32
    ).unsqueeze(0).unsqueeze(0).to(device)

    # Extract CNN embedding
    model.eval()

    with torch.no_grad():
        query_embedding = model.get_embedding(input_tensor)

    query_embedding = query_embedding.cpu().numpy()

    # Compare drawing to EVERY class
    results = []

    for class_name in categories:

        row = {
            "class": class_name
        }

        for k in [1, 5, 10]:
            centroids = centroids_by_k[k][class_name]

            similarities = cosine_similarity(
                query_embedding,
                centroids
            )

            row[f"k={k} average"] = round(
                float(similarities.mean()),
                4
            )

            row[f"k={k} max"] = round(
                float(similarities.max()),
                4
            )

        results.append(row)

    report = pd.DataFrame(results)

    # Highest similarity first
    report = report.sort_values(by="k=10 max", ascending=False)


interface = gr.Interface(
    fn=predict,
    inputs=gr.ImageEditor(type="pil"),
    outputs=gr.Dataframe(label="Cosine Similarity by Category")
)

interface.launch()
'''

'\nimport gradio as gr\n\ndef predict(editor_value):\n    if editor_value is None:\n        return pd.DataFrame()\n\n    img = editor_value["composite"]\n\n    if img is None:\n        return pd.DataFrame()\n\n    # Same preprocessing used for QuickDraw training images\n    # Preprocess Gradio drawing\n    img = img.convert("RGB")\n    img = img.resize(\n        (IMAGE_SIZE, IMAGE_SIZE),\n        Image.Resampling.LANCZOS\n    )\n\n    img_array = np.array(img).astype(np.float32)\n\n    # Convert colored/black strokes on white background\n    # into white strokes on black background\n    img_array = 1.0 - (img_array.min(axis=2) / 255.0)\n\n    input_tensor = torch.tensor(\n        img_array,\n        dtype=torch.float32\n    ).unsqueeze(0).unsqueeze(0).to(device)\n\n    # Extract CNN embedding\n    model.eval()\n\n    with torch.no_grad():\n        query_embedding = model.get_embedding(input_tensor)\n\n    query_embedding = query_embedding.cpu().numpy()\n\n    # Compare drawing to EVERY